# Statistics slice (Asif): return properties, why we forecast volatility, and stationarity
These cells only read `data/processed/model_table.csv` and only use the **training rows** (`split == "train"`, 2005-03-30 to 2014-12-23). Nothing here touches the test years.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os
import sys

if os.path.basename(os.getcwd()) == "econometric":   # same as the other slices: run from the repo root
    os.chdir("../..")
sys.path.append(os.getcwd())

TARGET = "y"                               # log of next-5-day realised volatility
FEATURES = [f"f{i}" for i in range(1, 11)]  # f1 ... f10
os.makedirs("results/figures", exist_ok=True)

table = pd.read_csv("data/processed/model_table.csv", parse_dates=["date"])
train = table[table["split"] == "train"].reset_index(drop=True)
r = train["f5"]                            # f5 = today's SPY log return (sign kept)
print("Training rows:", len(train), "| returns used:", len(r))

### Return properties
Daily returns have a mean near zero and very fat tails (excess kurtosis about 14, where a normal distribution has 0). Skewness is only slightly negative (about −0.09) because 2008 had very large up-days as well as falls.

In [ ]:
props = pd.Series({
    "mean (%)": r.mean() * 100,
    "std (%)": r.std() * 100,
    "annualised vol (%)": r.std() * np.sqrt(252) * 100,
    "skewness": stats.skew(r),
    "excess kurtosis": stats.kurtosis(r),   # 0 for a normal distribution
    "min (%)": r.min() * 100,
    "max (%)": r.max() * 100,
})
jb = stats.jarque_bera(r)
print(props.round(3).to_string())
print(f"Jarque-Bera = {jb.statistic:.1f}, p = {jb.pvalue:.2g}  (p < 0.05 -> returns are not normal)")

### Why volatility and not returns
If returns were predictable, their autocorrelation would be large. In our training years it is small (|ρ| ≤ 0.09) and switches sign from lag to lag, so it is too weak to forecast with. A few lags poke just outside the 95% band, mostly from the 2008 crisis. Absolute returns (how big the move was) have autocorrelation between 0.26 and 0.42 at every lag up to a month: big moves follow big moves. That is volatility clustering, and it is why we forecast volatility.

In [ ]:
lags = range(1, 23)                       # up to about one month
acf = pd.DataFrame({
    "lag": list(lags) * 2,
    "autocorrelation": [r.autocorr(k) for k in lags] + [r.abs().autocorr(k) for k in lags],
    "series": ["returns"] * len(lags) + ["absolute returns"] * len(lags),
})
band = 1.96 / np.sqrt(len(r))             # 95% band if there were no autocorrelation

sns.set_theme(style="whitegrid", font_scale=1.1)   # same as Main_Notebook
fig, ax = plt.subplots(figsize=(3.5, 2.8))
sns.lineplot(data=acf, x="lag", y="autocorrelation", hue="series", marker="o", ax=ax)
ax.axhspan(-band, band, color="grey", alpha=0.2)
ax.axhline(0, color="black", lw=0.8)
ax.set_title("Returns vs absolute returns\n(SPY, training years)", fontsize=11)
ax.set_xlabel("Lag (trading days)", fontsize=11)
ax.set_ylabel("Autocorrelation", fontsize=11)
ax.legend(fontsize=9, loc="center right")
fig.tight_layout()
fig.savefig("results/figures/acf_returns_vs_abs.png", dpi=300)
plt.show()

print(acf.pivot(index="lag", columns="series", values="autocorrelation").loc[[1, 5, 22]].round(3))
print(f"95% band: +/-{band:.3f}")

### Stationarity (ADF and KPSS)
The two tests have opposite null hypotheses, so we use both:
- **ADF**: null = non-stationary (unit root). Small p -> stationary.
- **KPSS**: null = stationary. Small p -> non-stationary.

A series is called stationary when ADF p < 0.05 **and** KPSS p > 0.05. KPSS p-values are only tabulated between 0.01 and 0.10, so statsmodels caps them there.

In [ ]:
import warnings
from statsmodels.tsa.stattools import adfuller, kpss

rows = []
for col in [TARGET] + FEATURES:
    x = train[col].dropna()
    adf_stat, adf_p, *_ = adfuller(x, autolag="AIC")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")    # warning when the p-value is outside the table
        kpss_stat, kpss_p, *_ = kpss(x, regression="c", nlags="auto")
    if adf_p < 0.05 and kpss_p > 0.05:
        verdict = "stationary"
    elif adf_p >= 0.05 and kpss_p <= 0.05:
        verdict = "non-stationary"
    else:
        verdict = "mixed"
    rows.append({"series": col, "ADF stat": adf_stat, "ADF p": adf_p,
                 "KPSS stat": kpss_stat, "KPSS p": kpss_p, "verdict": verdict})

stationarity = pd.DataFrame(rows).round(3)
stationarity.to_csv("results/stationarity_table.csv", index=False)
stationarity

**Reading the table:** write one line here once it has run, e.g. which series are "mixed". Volatility series are often *mixed*: ADF rejects a unit root, but KPSS also rejects because volatility is very persistent (long memory). That is not a problem for our models, it is exactly the persistence HAR uses. If any feature is clearly non-stationary, add a row to the "What we tried" table.